In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path


# Create a process data directory if it doesn't already exist
processed_data_path = Path("/Volumes/SANDISKUSBC/MLOps/hdb_price_prediction_mlops/data/processed")

In [ ]:
#functions


def readJson(file_path):
    try:
        # Open and read the JSON file
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
        # Print or process the data
        print(type(data))  # Typically <class 'dict'> or <class 'list'>
        print(data)
        return data
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
    except json.JSONDecodeError:
        print(f"Error: Failed to decode JSON from {file_path}.")



def getColTypes(df):
    categorical_cols = df[mycols].select_dtypes(include=['object']).columns
    numeric_cols = df[mycols].select_dtypes(include=['number']).columns
    return categorical_cols,numeric_cols


def setCategoryColsToDf(df,categorical_cols):
    for col in categorical_cols:
        df[col] = df[col].astype('category')
    return df


# Helper function for evaluation
def evaluate_metrics(y_true, y_pred):
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2





In [3]:

import mlflow
mlflow.set_tracking_uri('http://127.0.0.1:5000/')

logged_model = 'runs:/b6dd7a954dbe477387701422943dd2b9/model'

# Load model as a PyFuncModel.
#loaded_model = mlflow.pyfunc.load_model(logged_model)
loaded_model = mlflow.xgboost.load_model(logged_model)

In [6]:
hold_df= pd.read_csv(processed_data_path/'featured_hold.csv')

mycols = ['flat_type_rank', 'region_ura_rank', 'town_rank','storey_range_rank','flat_model_rank'
         ,'distance_to_cbd', 'floor_area_sqm', 'remaining_lease_years']
target = 'resale_price'

categorical_cols,numeric_cols = getColTypes(hold_df)
hold_df = setCategoryColsToDf(hold_df,categorical_cols)


X_hold = hold_df[mycols]
y_hold = hold_df[target]

X_hold.dtypes

flat_type_rank             int64
region_ura_rank            int64
town_rank                  int64
storey_range_rank          int64
flat_model_rank            int64
distance_to_cbd          float64
floor_area_sqm           float64
remaining_lease_years      int64
dtype: object

In [7]:
y_pred = loaded_model.predict(X_hold)
rmse, mae, r2 = evaluate_metrics(y_hold, y_pred)
hold_metric = {'data':'hold_df','MAE':mae,'RMSE':rmse,'R²':r2}
hold_metric

{'data': 'hold_df',
 'MAE': 122513.43092084394,
 'RMSE': 147061.38731618313,
 'R²': 0.4686890226030944}

In [ ]:
import json

# Define the paths for each map
flat_type_rank_map_path = processed_data_path / 'flat_type_rank_map.json'
region_rank_map_path = processed_data_path / 'region_rank_map.json'
town_rank_map_path = processed_data_path / 'town_rank_map.json'

flat_type_rank_map = readJson(flat_type_rank_map_path)
region_rank_map = readJson(region_rank_map_path)
town_rank_map = readJson(town_rank_map_path)


In [ ]:
hold_df['flat_type_rank']=hold_df['flat_type'].map(flat_type_rank_map).fillna(0)
hold_df['region_ura_rank']=hold_df['region_ura'].map(region_rank_map).fillna(0)
hold_df['town_rank'] = hold_df['town'].map(town_rank_map).fillna(0)